In [5]:
import torch
import torch.nn.functional as F
import os

def precompute_and_save(input_path, output_path):
    print(f"Loading data from {input_path}...")
    data_obj = torch.load(input_path, map_location="cpu")
    
    # --- FIXED DYNAMIC KEY DETECTION ---
    # We must ensure 'data' becomes a list, array, or tensor, NOT a dict.
    if isinstance(data_obj, dict):
        # Print keys so you can see exactly what's inside if it fails again
        print(f"Detected keys in dict: {list(data_obj.keys())}")
        
        # Try to find the data container. 
        # Adjust 'samples' to whichever key actually contains your numeric data.
        target_key = 'samples' if 'samples' in data_obj else list(data_obj.keys())[0]
        data = data_obj[target_key]
        print(f"Extracting data from key: '{target_key}'")
    else:
        data = data_obj

    # Now 'data' should be a sequence (list, numpy array, or tensor)
    if not isinstance(data, torch.Tensor):
        data = torch.FloatTensor(data)
    
    # Ensure (Batch, Channels, Time)
    if data.dim() == 2:
        data = data.unsqueeze(1)
        
    print(f"Detected Shape: {data.shape} | Processing...")
    
    n_samples, n_channels, n_time = data.shape
    time_list, fourier_list, wavelet_list = [], [], []
    window = torch.hann_window(128)
    
    # Process in chunks
    for i in range(0, n_samples, 500):
        batch = data[i:min(i+500, n_samples)]
        B, C, L = batch.shape
        batch_flat = batch.view(B * C, L)
        
        # 1. FFT
        x_fft = torch.fft.rfft(batch_flat, dim=-1)
        fourier_feat = torch.cat([torch.abs(x_fft), torch.angle(x_fft)], dim=-1)
        fourier_list.append(fourier_feat.view(B, C * fourier_feat.shape[-1]))
        
        # 2. STFT
        x_stft = torch.abs(torch.stft(batch_flat, n_fft=128, hop_length=64, window=window, return_complex=True))
        x_stft = x_stft.view(B, C, x_stft.shape[-2], x_stft.shape[-1])
        wavelet_feat = F.pad(x_stft[:, :, :64, :], (0, 1))
        wavelet_list.append(wavelet_feat)
        
        time_list.append(batch)
        if (i // 500) % 5 == 0: print(f"Progress: {i}/{n_samples}...")

    torch.save({'time': torch.cat(time_list), 
                'fourier': torch.cat(fourier_list), 
                'wavelet': torch.cat(wavelet_list)}, output_path)
    
    print("✅ Pre-computation complete!")

# Run the script
precompute_and_save(
    "/home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt", 
    "/home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_precomputed.pt"
)

Loading data from /home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt...
Detected keys in dict: ['train', 'val', 'test']
Extracting data from key: 'train'


TypeError: new(): data must be a sequence (got dict)